# Imports

In [1]:
import optuna
import pickle

import numpy as np
import pandas as pd

from utils import load_pickle
from xgboost import XGBClassifier

from sklearn.metrics import log_loss, balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

/home/junior/Documentos/GitHub/kaggle-competition-predicting-stellar-class/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
label_encoder = load_pickle('../models/label_encoder.pkl')

# Loading Datasets

In [3]:
X_train = pd.read_parquet('../data/X_train_stacking_layer_three.parquet')
y_train = pd.read_parquet('../data/y_train.parquet')

X_test = pd.read_parquet('../data/X_test_stacking_layer_three.parquet')

In [4]:
X_train.head()

,lgbm_0,lgbm_1,lgbm_2,cat_0,cat_1,cat_2,xgb_0,xgb_1,xgb_2,hist_0,hist_1,hist_2,rf_0,rf_1,rf_2,extra_0,extra_1,extra_2
0,0.999778,0.000212,0.000009,0.999787,0.000174,0.000040,0.999864,0.000121,0.000016,0.999733,0.000264,2.459617e-06,0.999775,0.000225,2.616556e-08,0.998912,0.000955,0.000133
1,0.994627,0.000176,0.005197,0.995247,0.000365,0.004388,0.992235,0.000199,0.007566,0.973059,0.000137,2.680453e-02,0.994737,0.000047,5.216709e-03,0.976808,0.000850,0.022342
2,0.000048,0.999944,0.000008,0.000483,0.999485,0.000032,0.000169,0.999822,0.000009,0.000015,0.999984,1.319155e-06,0.000011,0.999989,9.836632e-08,0.000068,0.999897,0.000035
3,0.999915,0.000081,0.000005,0.999745,0.000217,0.000039,0.999858,0.000127,0.000014,0.999960,0.000040,6.602657e-07,0.999793,0.000207,3.828407e-08,0.998919,0.000947,0.000134
4,0.998164,0.001812,0.000024,0.998143,0.001764,0.000093,0.998504,0.001459,0.000037,0.986426,0.013556,1.806930e-05,0.998227,0.001769,3.566911e-06,0.994430,0.005248,0.000322


In [5]:
X_test.head()

,lgbm_0,lgbm_1,lgbm_2,cat_0,cat_1,cat_2,xgb_0,xgb_1,xgb_2,hist_0,hist_1,hist_2,rf_0,rf_1,rf_2,extra_0,extra_1,extra_2
0,0.998177,0.001784,0.000039,0.997654,0.002153,0.000193,0.998108,0.001843,0.000049,0.997980,0.002008,0.000012,0.998183,0.001800,0.000017,0.993598,0.005832,0.000569
1,0.997269,0.002710,0.000021,0.997945,0.001978,0.000077,0.997185,0.002778,0.000037,0.994660,0.005324,0.000015,0.997905,0.002092,0.000003,0.993669,0.005970,0.000361
2,0.996089,0.000609,0.003303,0.997698,0.000711,0.001592,0.997502,0.000382,0.002115,0.997295,0.001499,0.001207,0.997506,0.000715,0.001780,0.988872,0.002584,0.008544
3,0.000643,0.000107,0.999250,0.001658,0.000165,0.998177,0.000730,0.000132,0.999138,0.000139,0.000020,0.999841,0.000541,0.000023,0.999436,0.000378,0.000214,0.999409
4,0.999631,0.000357,0.000012,0.999578,0.000365,0.000058,0.999737,0.000245,0.000018,0.999526,0.000469,0.000005,0.999785,0.000205,0.000010,0.998670,0.001116,0.000214


# Machine Learning

In [6]:
def objective(trial, X, y):

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    scores = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):

        X_train_fold = X.iloc[train_idx, :]
        X_valid_fold = X.iloc[valid_idx, :]

        y_train_fold = y.iloc[train_idx]
        y_valid_fold = y.iloc[valid_idx]

        model = XGBClassifier(
            objective='multi:softprob',
            num_class=len(np.unique(y)),
            random_state=42,
            n_jobs=1,
            n_estimators=trial.suggest_int('n_estimators', 50, 500),
            learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            max_depth=trial.suggest_int('max_depth', 2, 8),
            min_child_weight=trial.suggest_int('min_child_weight', 1, 20),
            gamma=trial.suggest_float('gamma', 0, 10),
            subsample=trial.suggest_float('subsample', 0.6, 1.0),
            colsample_bytree=trial.suggest_float('colsample_bytree', 0.6, 1.0),
            reg_alpha=trial.suggest_float('reg_alpha', 1e-5, 10, log=True),
            reg_lambda=trial.suggest_float('reg_lambda', 1e-5, 10, log=True),
            eval_metric='mlogloss'
        ).fit(X_train_fold, y_train_fold)

        proba = model.predict_proba(X_valid_fold)

        w0 = trial.suggest_float('weight_class_0', 0.1, 2.0)
        w1 = trial.suggest_float('weight_class_1', 0.1, 2.0)
        w2 = trial.suggest_float('weight_class_2', 0.1, 2.0)

        weighted_probas = proba * np.array([w0, w1, w2])

        pred = np.argmax(weighted_probas, axis=1)

        score = balanced_accuracy_score(y_valid_fold, pred)

        scores.append(score)

        trial.report(np.mean(scores), step=fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(scores)


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42), pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study.optimize(lambda trial: objective(trial, X_train, y_train.class_encoded), n_trials=60, n_jobs=-1, show_progress_bar=True)

print("Best trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-06-18 11:08:30,287] A new study created in memory with name: no-name-5a0dd625-0a1f-41a3-8870-78863b9f5b88
Best trial: 5. Best value: 0.960732:   2%|██▎                                                                                                                                     | 1/60 [03:10<3:07:04, 190.25s/it]

[I 2026-06-18 11:11:40,524] Trial 5 finished with value: 0.9607315870670836 and parameters: {'n_estimators': 224, 'learning_rate': 0.22521094116193066, 'max_depth': 8, 'min_child_weight': 7, 'gamma': 2.24831537039333, 'subsample': 0.7134768081733256, 'colsample_bytree': 0.6990877331228785, 'reg_alpha': 0.0028507607164606026, 'reg_lambda': 2.7733432614101884e-05, 'weight_class_0': 0.8409983621843435, 'weight_class_1': 0.5231612023943638, 'weight_class_2': 1.808966311077602}. Best is trial 5 with value: 0.9607315870670836.


Best trial: 5. Best value: 0.960732:   3%|████▌                                                                                                                                    | 2/60 [03:10<1:15:52, 78.49s/it]

[I 2026-06-18 11:11:40,779] Trial 1 finished with value: 0.9421088763253993 and parameters: {'n_estimators': 177, 'learning_rate': 0.05877561790082838, 'max_depth': 4, 'min_child_weight': 13, 'gamma': 5.973589618905523, 'subsample': 0.7501305941654245, 'colsample_bytree': 0.9349688350459817, 'reg_alpha': 0.0006511864552189196, 'reg_lambda': 3.709764390617972, 'weight_class_0': 1.6536056885915276, 'weight_class_1': 0.8279337472043805, 'weight_class_2': 0.5875819148653673}. Best is trial 5 with value: 0.9607315870670836.


Best trial: 5. Best value: 0.960732:   5%|██████▉                                                                                                                                    | 3/60 [03:43<54:52, 57.76s/it]

...

[I 2026-06-18 11:26:50,413] Trial 56 finished with value: 0.9659712299623262 and parameters: {'n_estimators': 208, 'learning_rate': 0.08136104775012606, 'max_depth': 8, 'min_child_weight': 15, 'gamma': 2.5630252305740404, 'subsample': 0.9537146361391414, 'colsample_bytree': 0.9976356705136931, 'reg_alpha': 0.0007975797053193805, 'reg_lambda': 1.2933546735460966, 'weight_class_0': 0.24075207391881165, 'weight_class_1': 0.4414136438892312, 'weight_class_2': 0.8333749787669058}. Best is trial 43 with value: 0.9664552616525155.


Best trial: 43. Best value: 0.966455:  98%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋  | 59/60 [18:23<00:07,  7.09s/it]

[I 2026-06-18 11:26:53,784] Trial 55 finished with value: 0.9660709781009856 and parameters: {'n_estimators': 201, 'learning_rate': 0.07849806913528629, 'max_depth': 7, 'min_child_weight': 11, 'gamma': 1.6769777010022135, 'subsample': 0.8917550056442071, 'colsample_bytree': 0.9833762509685984, 'reg_alpha': 0.0006117260861332508, 'reg_lambda': 1.7983937678611819, 'weight_class_0': 0.25573122941141085, 'weight_class_1': 0.749484224565127, 'weight_class_2': 0.8561175719159196}. Best is trial 43 with value: 0.9664552616525155.


Best trial: 43. Best value: 0.966455: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 60/60 [18:30<00:00, 18.51s/it]

[I 2026-06-18 11:27:00,677] Trial 58 finished with value: 0.9651587235262434 and parameters: {'n_estimators': 198, 'learning_rate': 0.01010421961774935, 'max_depth': 4, 'min_child_weight': 15, 'gamma': 2.6272740636438927, 'subsample': 0.9534464955231017, 'colsample_bytree': 0.6751163998580703, 'reg_alpha': 0.0009033520505399793, 'reg_lambda': 0.037293837479703225, 'weight_class_0': 0.26702682713470505, 'weight_class_1': 0.6961526937105991, 'weight_class_2': 0.77239030195707}. Best is trial 43 with value: 0.9664552616525155.
Best trial score:
0.9664552616525155

Best params:
{'n_estimators': 158, 'learning_rate': 0.01984170688781394, 'max_depth': 6, 'min_child_weight': 16, 'gamma': 1.3382056143627181, 'subsample': 0.932750143000414, 'colsample_bytree': 0.7865877091043509, 'reg_alpha': 0.00024307656295547938, 'reg_lambda': 0.01538631782677991, 'weight_class_0': 0.19020754781253635, 'weight_class_1': 0.8360460311543438, 'weight_class_2': 0.9956026282725565}


In [19]:
study.optimize(lambda trial: objective(trial, X_train, y_train.class_encoded), n_trials=60, n_jobs=-1, show_progress_bar=True)

Best trial: 105. Best value: 0.966562:   2%|██▎                                                                                                                                    | 1/60 [01:27<1:25:59, 87.46s/it]

[I 2026-06-18 14:29:17,339] Trial 125 finished with value: 0.9652497460235324 and parameters: {'n_estimators': 88, 'learning_rate': 0.010583183806081309, 'max_depth': 2, 'min_child_weight': 2, 'gamma': 6.922689857485631, 'subsample': 0.7124992373435034, 'colsample_bytree': 0.6934580469618686, 'reg_alpha': 0.2710136085428937, 'reg_lambda': 0.002196057815042488, 'weight_class_0': 0.2957567632084959, 'weight_class_1': 1.7117684980760144, 'weight_class_2': 1.7797466488756442}. Best is trial 105 with value: 0.9665620964645611.


Best trial: 105. Best value: 0.966562:   3%|████▌                                                                                                                                    | 2/60 [01:41<43:01, 44.51s/it]

[I 2026-06-18 14:29:31,786] Trial 124 pruned. 


Best trial: 105. Best value: 0.966562:   5%|██████▊                                                                                                                                  | 3/60 [01:59<30:41, 32.30s/it]

...

[I 2026-06-18 14:39:13,913] Trial 174 finished with value: 0.9664650204663532 and parameters: {'n_estimators': 220, 'learning_rate': 0.017574095319734112, 'max_depth': 3, 'min_child_weight': 4, 'gamma': 3.771084157475358, 'subsample': 0.7923860357345857, 'colsample_bytree': 0.6134420379930517, 'reg_alpha': 0.017703385589318266, 'reg_lambda': 0.077105964503766, 'weight_class_0': 0.3192411249959975, 'weight_class_1': 1.4226420701004745, 'weight_class_2': 1.829096495393345}. Best is trial 105 with value: 0.9665620964645611.


Best trial: 105. Best value: 0.966562:  98%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋  | 59/60 [11:32<00:08,  8.73s/it]

[I 2026-06-18 14:39:22,734] Trial 175 finished with value: 0.9663647533996773 and parameters: {'n_estimators': 210, 'learning_rate': 0.01726825185342794, 'max_depth': 3, 'min_child_weight': 4, 'gamma': 0.9896322552721957, 'subsample': 0.6652749412517006, 'colsample_bytree': 0.6792365334487718, 'reg_alpha': 0.017934414163542155, 'reg_lambda': 0.00013294399769766415, 'weight_class_0': 0.29393367832695083, 'weight_class_1': 1.4197091826529022, 'weight_class_2': 1.8763038108188155}. Best is trial 105 with value: 0.9665620964645611.


Best trial: 105. Best value: 0.966562: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 60/60 [12:18<00:00, 12.31s/it]

[I 2026-06-18 14:40:08,397] Trial 162 finished with value: 0.9664271738032311 and parameters: {'n_estimators': 466, 'learning_rate': 0.014385708378549225, 'max_depth': 3, 'min_child_weight': 7, 'gamma': 1.5451503514541343, 'subsample': 0.7681544490170222, 'colsample_bytree': 0.6350802409209809, 'reg_alpha': 0.05160734481760502, 'reg_lambda': 3.200265374372273e-05, 'weight_class_0': 0.3278220997226016, 'weight_class_1': 1.2931850974460883, 'weight_class_2': 1.9583199498222164}. Best is trial 105 with value: 0.9665620964645611.


In [20]:
xgb_params = {k: v for k, v in study.best_params.items() if k not in ['weight_class_0', 'weight_class_1', 'weight_class_2']}

xgb = XGBClassifier(
    **xgb_params,
    objective='multi:softprob',
    num_class=3,
    random_state=42,
    n_jobs=-1,
    eval_metric='mlogloss'
).fit(X_train, y_train.class_encoded)

test_proba = xgb.predict_proba(X_test)

weights = np.array([study.best_params['weight_class_0'], study.best_params['weight_class_1'], study.best_params['weight_class_2']])
weighted_probas = test_proba * weights

pred = np.argmax(weighted_probas, axis=1)

In [21]:
sub_labels = label_encoder.inverse_transform(pred)

# Submission

In [22]:
submission = pd.read_csv('../data/sample_submission.csv')
submission['class'] = sub_labels

submission.to_csv('../data/submission_stacking_xgb.csv', index=False)

In [23]:
submission.head()

,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY


In [24]:
X_train.columns

Index(['lgbm_0', 'lgbm_1', 'lgbm_2', 'cat_0', 'cat_1', 'cat_2', 'xgb_0',
       'xgb_1', 'xgb_2', 'hist_0', 'hist_1', 'hist_2', 'rf_0', 'rf_1', 'rf_2',
       'extra_0', 'extra_1', 'extra_2'],
      dtype='str')